<a href="https://colab.research.google.com/github/MNawfal24/MachineLearning/blob/main/Minggu6_ANN/MNawfalMA_JS06_ANN_P3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679802 sha256=f5ae260a6d72228d8ae58002ff4f5f37cf700b387bf39e473d2ce444b956ccca
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [ ]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

# ===========================
# 1. Buat data 2D acak
# ===========================
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # cari 5 tetangga terdekat

# ===========================
# 2. Exact NN (Brute Force)
# ===========================
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

# ===========================
# 3. HNSW
# ===========================
# Inisialisasi index HNSW
p = hnswlib.Index(space='l2', dim=dim)

# Ukuran maksimum elemen yang bisa ditampung
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Tambahkan data
p.add_items(data)

# Set parameter pencarian
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "detik")


=== Exact NN ===
Indices: [[172 729 950 446   2]]
Distances: [[0.00840982 0.0217384  0.03362996 0.03704155 0.04175609]]
Waktu: 0.05798649787902832 detik

=== HNSW ===
Indices: [[172 729 950 446   2]]
Distances: [[7.0725095e-05 4.7255782e-04 1.1309744e-03 1.3720762e-03 1.7435713e-03]]
Waktu: 0.0001628398895263672 detik


In [ ]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Skenario eksperimen sesuai tugas:
data_sizes = [1000, 1000000]
dimensions = [2, 5]
# Format metrik: (sklearn_metric, hnsw_space)
metrics = [('euclidean', 'l2'), ('cosine', 'cosine')]

hasil_eksperimen = []
k = 5

print("Sedang memproses eksperimen (1 juta data mungkin butuh waktu beberapa saat)...\n")

for num_elements in data_sizes:
    for dim in dimensions:
        for metric_sk, metric_hn in metrics:

            # Buat data acak sesuai iterasi saat ini
            data = np.random.random((num_elements, dim)).astype(np.float32)
            query = np.random.random((1, dim)).astype(np.float32)

            # Hitung Exact NN
            nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric=metric_sk)
            nn.fit(data)
            start_ex = time.time()
            nn.kneighbors(query)
            time_ex = time.time() - start_ex

            # Hitung HNSW
            p = hnswlib.Index(space=metric_hn, dim=dim)
            p.init_index(max_elements=num_elements, ef_construction=100, M=16)
            p.add_items(data)
            p.set_ef(50)

            start_hn = time.time()
            p.knn_query(query, k=k)
            time_hn = time.time() - start_hn

            # Simpan baris ke tabel
            hasil_eksperimen.append({
                "Jumlah Data": num_elements,
                "Dimensi": f"{dim}D",
                "Metrik": metric_sk.capitalize(),
                "Waktu Exact NN (s)": round(time_ex, 6),
                "Waktu HNSW (s)": round(time_hn, 6),
                "Status": f"HNSW {round(time_ex / time_hn, 1)}x lebih cepat" if time_hn > 0 else "N/A"
            })

# Tampilkan hasil akhir dalam bentuk tabel dataframe
df_hasil = pd.DataFrame(hasil_eksperimen)
from IPython.display import display, HTML
display(HTML(df_hasil.to_html(index=False)))

Sedang memproses eksperimen (1 juta data mungkin butuh waktu beberapa saat)...



Jumlah Data,Dimensi,Metrik,Waktu Exact NN (s),Waktu HNSW (s),Status
1000,2D,Euclidean,0.008886,0.000230,HNSW 38.7x lebih cepat
1000,2D,Cosine,0.023721,0.000114,HNSW 208.6x lebih cepat
1000,5D,Euclidean,0.002871,0.000123,HNSW 23.4x lebih cepat
1000,5D,Cosine,0.004599,0.000139,HNSW 33.1x lebih cepat
1000000,2D,Euclidean,0.052024,0.000143,HNSW 364.3x lebih cepat
1000000,2D,Cosine,0.037565,0.000293,HNSW 128.4x lebih cepat
1000000,5D,Euclidean,0.054470,0.000190,HNSW 286.7x lebih cepat
1000000,5D,Cosine,0.065752,0.000193,HNSW 341.3x lebih cepat
